# Track simulation with Neptune.ai with setup from Custom Algorithms Implementations tutorial.

> In this notebook, we introduce tools for online tracking of experiments. We use `Two Route Yeild (TRY)` network and simulate **10 human agents** for `2000 days`. After 100 days **10 of the human agents** mutate into automated vehicles (AVs) and use the [`MAPPO`]((https://www.nature.com/articles/nature14236)) (Independent Q-Learning) algorithm. The AVs are `selfish` and their goal is to minimize their own travel time. We will present from scratch connection to experiments overview and analysis via Neptune app.

#### Import necessary libraries for simulation

In [2]:
from tqdm import tqdm
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../')))

#from extended_mutation import MyExtendedMutation
from routerl import TrafficEnvironment
from routerl import MAPPO
from routerl import Keychain as kc

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

### Import Neptune

In [19]:
import neptune

#### Initialize Neptune

Replace variables:

- `project` with path to your own project on the Neptune workspace,

- `name` with current run custom name,

- `tags` with keywords for specific run,

- `api_token` with your personal token generated via Neptune account

In [ ]:
neptune_run = neptune.init_run(
    project="neptune_workspace/project_name",
    name="mappo",
    tags=["AVs", "MARL", "TRY network"],
    api_token="your api_token",
)

#### Hyperparameters setting


In [3]:
new_machines_after_mutation = 10
human_learning_episodes = 1
training_episodes = 2000
testing_episodes = 100

total_episodes = human_learning_episodes + training_episodes

env_params = {
    "agent_parameters" : {
        "new_machines_after_mutation": new_machines_after_mutation,

        "human_parameters" :
        {
            "model" : "general_model",

            "noise_weight_agent" : 0,
            "noise_weight_path" : 0.8,
            "noise_weight_day" : 0.2,

            "beta" : -1,
            "beta_k_i_variability" : 0.1,
            "epsilon_i_variability" : 0.1,
            "epsilon_k_i_variability" : 0.1,
            "epsilon_k_i_t_variability" : 0.1,

            "greedy" : 0.9,
            "gamma_c" : 0.0,
            "gamma_u" : 0.0,
            "remember" : 1,

            "alpha_zero" : 0.8,
            "alphas" : [0.2]
        },
        "machine_parameters" :
        {
            "behavior" : "selfish",
            "observation_type" : "previous_agents_plus_start_time",
        }
    },
    "simulator_parameters" : {
        "network_name" : "two_route_yield",
        "sumo_type" : "sumo",
    },
    "plotter_parameters" : {
        "phases" : [0, human_learning_episodes, int(training_episodes) + human_learning_episodes],
        "smooth_by" : 50,
        "phase_names" : [
            "Human learning",
            "Mutation - Machine learning",
            "Testing phase"
        ],
        "plot_choices": "basic"
    },
    "path_generation_parameters":
    {
        "number_of_paths" : 2,
        "beta" : -1,
        "visualize_paths" : True
    }
}

#### Set parameters to Neptune app

While all hyperparams are set up, you can pass them to Neptune in order to have richer description in online version of simulation

In [ ]:
neptune_run["parameters"] = env_params

#### Environment initialization


> In this example, the environment initially contains only human agents.


> If the paths are already created then create_paths=False, we don't have to create again.

In [20]:
env = TrafficEnvironment(seed=42, create_agents=True, create_paths=True, marginal_cost_calculation=False, **env_params)

PermissionError: [WinError 32] Proces nie może uzyskać dostępu do pliku, ponieważ jest on używany przez inny proces: 'training_records\\SUMO_output\\detailed_sumo_stats_3.xml'

In [21]:
print("Number of total agents is: ", len(env.all_agents), "\n")
print("Number of human agents is: ", len(env.human_agents), "\n")
print("Number of machine agents (autonomous vehicles) is: ", len(env.machine_agents), "\n")

Number of total agents is:  100 

Number of human agents is:  90 

Number of machine agents (autonomous vehicles) is:  10 



In [22]:
env.start()

TraCIException: Connection '654' is already active.

In [10]:
env.reset()

({}, {})

In [11]:
pbar = tqdm(total=total_episodes, desc="Human learning")

for episode in range(human_learning_episodes):
    env.step()

Human learning:   0%|          | 0/2001 [00:00<?, ?it/s]

#### Mutation

> **Mutation**: a portion of human agents are converted into machine agents (autonomous vehicles).

In [23]:
pre_mutation_agents = env.all_agents.copy()

env.mutation_odd_id_agents()

> Humans are fixed to action zero in this case.

In [24]:
for human in env.human_agents:
    human.default_action = 0

In [25]:
machines = env.machine_agents.copy()
mutated_humans = dict()
for machine in machines:
    for human in pre_mutation_agents:
        if human.id == machine.id:
            mutated_humans[str(machine.id)] = human
            break

> Each AV agent has an independent learning component.

In [26]:
free_flows = env.get_free_flow_times()
for h_id, human in mutated_humans.items():
    initial_knowledge = free_flows[(human.origin, human.destination)]
    initial_knowledge = [0, 0]
    mutated_humans[h_id].model = MAPPO(3, len(initial_knowledge))

In [27]:
mutated_humans

{'3': Machine 3,
 '5': Machine 5,
 '7': Machine 7,
 '9': Machine 9,
 '11': Machine 11,
 '13': Machine 13,
 '15': Machine 15,
 '17': Machine 17,
 '19': Machine 19,
 '21': Machine 21,
 '4': Human 4,
 '8': Human 8,
 '12': Human 12,
 '16': Human 16,
 '20': Human 20,
 '23': Human 23,
 '25': Human 25,
 '27': Human 27,
 '29': Human 29,
 '31': Human 31}

#### AV's training loop

By specifying path for each agent, you can track rewards in each episode of training loop

In [29]:
pbar.set_description("AV learning")
os.makedirs("plots", exist_ok=True)
for episode in range(training_episodes):
    env.reset()
    for agent in env.agent_iter():
        observation, reward, termination, truncation, info = env.last()

        if termination or truncation:
            obs = [{kc.AGENT_ID : int(agent), kc.TRAVEL_TIME : -reward}]
            last_action = mutated_humans[agent].last_action
            last_observation = mutated_humans[agent].last_obs

            mutated_humans[agent].learn(last_action, obs)
            action = None
        else:
            action = mutated_humans[agent].act(observation)
            mutated_humans[agent].last_action = action
        # Transfer reward to Neptune
        neptune_run[f"av_training/{int(agent)}/reward"].append(reward)
        env.step(action)
    """if episode % 50 == 0:
        env.plot_results()"""
    pbar.update()

AV learning:   0%|          | 0/2001 [24:23<?, ?it/s]

NameError: name 'neptune_run' is not defined

#### AV's testing loop

The same thing can be obtained in any other place, like testing loop

In [ ]:
pbar.set_description("Testing")
for episode in range(testing_episodes):
    env.reset()
    for agent in env.agent_iter():
        observation, reward, termination, truncation, info = env.last()
        if termination or truncation:
            action = None
        else:
            action = mutated_humans[agent].act(observation)
        # Transfer reward to Neptune
        neptune_run[f"av_testing/{int(agent)}/reward"].append(reward)
        env.step(action)
    pbar.update()

pbar.close()
#os.makedirs(plots_folder, exist_ok=True)

>  Check `\plots` directory to find the plots created from this experiment.

In [ ]:
env.plot_results()

> Interrupt the connection with `SUMO`.

In [ ]:
env.stop_simulation()